# RAFT Pipeline on CUAD (Teacher-Student Knowledge Distillation)

A single, resumable, Kaggle-GPU (2x T4, 32GB total) notebook that builds a
Retrieval-Augmented Fine-Tuning (RAFT) dataset from the CUAD legal contracts
dataset and distills it into a small student model.

**Design notes / deviations from a generic RAFT recipe (read before running):**

- **Teacher model:** `unsloth/gemma-2-9b-it-bnb-4bit` (not the 27B variant).
  A 27B model in 4-bit is ~16GB of weights alone, which does not fit with headroom
  on a single 16GB T4, and Kaggle's 2x T4 has **no tensor parallelism** — a naive
  `device_map="auto"` split runs GPUs sequentially, layer-by-layer, so nothing is
  actually parallel and generation throughput does not improve (Unsloth's own
  model cards note 1xT4 is ~5x faster than 2xT4 for exactly this reason). The 9B
  variant fits comfortably on one T4 with room to spare, generates far faster,
  and leaves the second T4 free.
- **Unsloth is single-GPU only in the free/OSS tier.** Multi-GPU training/inference
  requires Unsloth Pro. So `FastLanguageModel` (Unsloth) is used **only** for the
  3B student in Phase 4 (which fits on one T4). The teacher (Phase 2/5) and the
  base/RAFT student during **evaluation** (Phase 5) are loaded with plain
  `transformers` + `bitsandbytes` instead, using `device_map="auto"` per the
  OOM-prevention rule, since inference doesn't require Unsloth's training kernels.
- **Ragas judge model:** rather than having the just-fine-tuned RAFT student score
  its own outputs (circular / biased), Faithfulness and Answer Relevancy are
  computed with the **teacher model reloaded as an independent judge**, wrapped
  via `LangchainLLMWrapper` per Ragas' custom-LLM integration path.
- **Rank-1/3/5 Exact Match & F1** are computed by retrieving the top-{1,3,5}
  chunks via FAISS and generating an answer conditioned on that context. The
  zero-shot config receives no retrieved context by construction, so its
  Rank-1/3/5 scores are identical (repeated) across the three columns — this is
  expected, not a bug. "Span F1 Score" and the Ragas metrics are all reported at
  k=3 as the representative retrieval depth.
- Every phase checks Hugging Face Hub for its output file first and skips
  recomputation if found (`HfApi().file_exists`). All intermediates and the final
  adapter are pushed to the Hub — nothing depends on `/kaggle/working` surviving
  a kernel restart.

**Prerequisites you must have in place before running:**
1. A Kaggle secret named `HF_TOKEN` (Add-ons → Secrets) — a Hugging Face token
   with **write** access.
2. That HF account must have accepted the Gemma license at
   `https://huggingface.co/google/gemma-2-9b-it` (gated model — ungated tokens
   will get a 401/403 on load).
3. Kaggle GPU accelerator enabled (2x T4).


In [4]:
# !pip install -q -U --force-reinstall datasets==2.21.0
# !pip install -q protobuf>=4.25.3
# !pip install -q -U unsloth bitsandbytes "transformers>=4.44" "trl>=0.12" peft \
#     sentence-transformers faiss-cpu "ragas>=0.2,<0.3" huggingface_hub accelerate \
#     langchain langchain-huggingface langchain-community tabulate pyarrow

In [5]:
import datasets; print(datasets.__version__)

4.3.0


In [6]:
# Cell 2: Imports, Configuration, HF Login, Shared Helpers
import os, gc, sys, json, time, random, logging, traceback, string, re, collections
import numpy as np
import pandas as pd
import torch
import faiss

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", stream=sys.stdout)
logger = logging.getLogger("RAFT-Pipeline")

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- Pipeline-wide parameters ----
CHUNK_SIZE_CHARS = 2000          # ~512 tokens
CHUNK_OVERLAP_CHARS = 250        # ~64 tokens
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEACHER_MODEL_NAME = "unsloth/gemma-2-9b-it-bnb-4bit"   # swapped from 27B, see notebook intro
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
STUDENT_MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
N_SYNTHETIC_SAMPLES = 500
N_EVAL_SAMPLES = 50
N_DISTRACTORS = 2
GROUNDING_THRESHOLD = 0.5
BATCH_CHECKPOINT_EVERY = 50

EMBEDDED_CHUNKS_FILE = "cuad_embedded_chunks.parquet"
RAW_SYNTHETIC_FILE = "cuad_raw_synthetic.parquet"
FILTERED_RAFT_FILE = "cuad_filtered_raft.parquet"
EVAL_HOLDOUT_FILE = "cuad_eval_holdout.parquet"
EVAL_RESULTS_FILE = "evaluation_results.csv"

WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)

from huggingface_hub import HfApi, login, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not read HF_TOKEN from Kaggle Secrets. Add a secret named 'HF_TOKEN' "
        "under Add-ons > Secrets with a Hugging Face WRITE token, and make sure that "
        "account has accepted the Gemma license at "
        "https://huggingface.co/google/gemma-2-9b-it before running this notebook."
    ) from e

login(token=hf_token)
api = HfApi()
HF_USER = api.whoami(token=hf_token)["name"]
HF_REPO_ID = f"{HF_USER}/cuad-raft-pipeline"
MODEL_REPO_ID = f"{HF_USER}/qwen2.5-3b-cuad-raft"

api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
api.create_repo(repo_id=MODEL_REPO_ID, repo_type="model", exist_ok=True)

logger.info(f"HF user: {HF_USER}")
logger.info(f"Dataset checkpoint repo: {HF_REPO_ID}")
logger.info(f"Model output repo: {MODEL_REPO_ID}")


def is_phase_complete(filename: str, repo_id: str = HF_REPO_ID, repo_type: str = "dataset") -> bool:
    '''Mandatory resume check: query HF Hub before doing any computation.'''
    try:
        return api.file_exists(repo_id=repo_id, filename=filename, repo_type=repo_type)
    except Exception as e:
        logger.warning(f"file_exists check failed for {filename}: {e}")
        return False


def push_parquet(df: pd.DataFrame, filename: str):
    local_path = os.path.join(WORKDIR, filename)
    df.to_parquet(local_path, index=False)
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=filename,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        commit_message=f"Add/update {filename}",
    )
    logger.info(f"Pushed {filename} ({len(df)} rows) to {HF_REPO_ID}")


def pull_parquet(filename: str) -> pd.DataFrame:
    local_path = hf_hub_download(repo_id=HF_REPO_ID, filename=filename, repo_type="dataset")
    df = pd.read_parquet(local_path)
    logger.info(f"Pulled {filename} ({len(df)} rows) from {HF_REPO_ID}")
    return df


def clear_cuda_cache_and_log():
    '''Call AFTER del <objects> in the calling cell's own scope -- del must
    happen in the cell itself so the real references are dropped, not just a
    helper function's local parameter names.'''
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            logger.info(f"  GPU {i}: {free / 1e9:.2f} GB free / {total / 1e9:.2f} GB total")
    logger.info("Memory cleanup complete.")


def normalize_answer(s: str) -> str:
    s = (s or "").lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def exact_match(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

logger.info("Config, HF login, and shared helpers ready.")


2026-07-25 19:44:40,125 | INFO | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-25 19:44:40,193 | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-07-25 19:44:40,260 | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-07-25 19:44:40,333 | INFO | HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 409 Conflict"
2026-07-25 19:44:40,408 | INFO | HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 409 Conflict"
2026-07-25 19:44:40,409 | INFO | HF user: abhifdsdf
2026-07-25 19:44:40,410 | INFO | Dataset checkpoint repo: abhifdsdf/cuad-raft-pipeline
2026-07-25 19:44:40,410 | INFO | Model output repo: abhifdsdf/qwen2.5-3b-cuad-raft
2026-07-25 19:44:40,412 | INFO | Config, HF login, and shared helpers ready.


## PHASE 1: Dataset Preparation & Embedding Indexing

Loads `theatticusproject/cuad-qa`, deduplicates contract passages, performs
sliding-window chunking, embeds with MiniLM, checkpoints to
`cuad_embedded_chunks.parquet`, and builds an in-RAM FAISS `IndexFlatIP` index.


In [7]:
# Phase 1: Dataset Preparation & Embedding Indexing
try:
    logger.info("=== PHASE 1: Dataset Preparation & Embedding Indexing ===")

    if is_phase_complete(EMBEDDED_CHUNKS_FILE):
        logger.info(f"{EMBEDDED_CHUNKS_FILE} found on {HF_REPO_ID}. Skipping chunking/embedding computation.")
        chunks_df = pull_parquet(EMBEDDED_CHUNKS_FILE)
    else:
        logger.info("No checkpoint found. Loading CUAD and building chunks from scratch...")
        from datasets import load_dataset

        cuad_raw = load_dataset("theatticusproject/cuad-qa", split="train")

        # Many QA rows share the same underlying contract passage -- dedupe on context text.
        seen = set()
        unique_docs = []
        for row in cuad_raw:
            ctx = row["context"]
            if ctx not in seen:
                seen.add(ctx)
                unique_docs.append(ctx)
        logger.info(f"Extracted {len(unique_docs)} unique contract passages from CUAD.")

        def sliding_window_chunks(text, size=CHUNK_SIZE_CHARS, overlap=CHUNK_OVERLAP_CHARS):
            chunks = []
            step = max(size - overlap, 1)
            for start in range(0, len(text), step):
                chunk = text[start:start + size]
                if len(chunk.strip()) > 0:
                    chunks.append(chunk)
                if start + size >= len(text):
                    break
            return chunks

        records = []
        for doc_id, doc_text in enumerate(unique_docs):
            for chunk_text in sliding_window_chunks(doc_text):
                records.append({"source_doc_id": doc_id, "text": chunk_text})

        chunk_df_raw = pd.DataFrame(records)
        chunk_df_raw["chunk_id"] = chunk_df_raw.index
        logger.info(
            f"Generated {len(chunk_df_raw)} chunks "
            f"(size={CHUNK_SIZE_CHARS} chars, overlap={CHUNK_OVERLAP_CHARS} chars)."
        )

        from sentence_transformers import SentenceTransformer

        embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
        embeddings = embed_model.encode(
            chunk_df_raw["text"].tolist(),
            batch_size=128,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype(np.float32)

        chunk_df_raw["embedding"] = embeddings.tolist()  # store as plain python lists for parquet
        chunks_df = chunk_df_raw[["chunk_id", "source_doc_id", "text", "embedding"]].reset_index(drop=True)

        push_parquet(chunks_df, EMBEDDED_CHUNKS_FILE)

        del embed_model
        clear_cuda_cache_and_log()

    # ---- Build FAISS IndexFlatIP in RAM (always rebuilt in-memory; never persisted to disk) ----
    embedding_matrix = np.vstack(
        chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy()
    )
    faiss_index = faiss.IndexFlatIP(embedding_matrix.shape[1])
    faiss_index.add(embedding_matrix)
    logger.info(f"FAISS IndexFlatIP built with {faiss_index.ntotal} vectors (dim={embedding_matrix.shape[1]}).")

except Exception:
    logger.error("Phase 1 failed.")
    logger.error(traceback.format_exc())
    raise


2026-07-25 19:44:40,423 | INFO | === PHASE 1: Dataset Preparation & Embedding Indexing ===
2026-07-25 19:44:40,494 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_embedded_chunks.parquet "HTTP/1.1 302 Found"
2026-07-25 19:44:40,495 | INFO | cuad_embedded_chunks.parquet found on abhifdsdf/cuad-raft-pipeline. Skipping chunking/embedding computation.
2026-07-25 19:44:40,569 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_embedded_chunks.parquet "HTTP/1.1 302 Found"


cuad_embedded_chunks.parquet: reconstructing file:   0%|          |  0.00B / 40.4MB            

cuad_embedded_chunks.parquet: downloading bytes:           |  0.00B            

2026-07-25 19:44:42,528 | INFO | Pulled cuad_embedded_chunks.parquet (12726 rows) from abhifdsdf/cuad-raft-pipeline
2026-07-25 19:44:42,585 | INFO | FAISS IndexFlatIP built with 12726 vectors (dim=384).


## PHASE 2: Synthetic Data Generation (Teacher Model)

Samples chunks, prompts the teacher (`unsloth/gemma-2-9b-it-bnb-4bit`) to write
one legal Q/A pair per chunk, assembles RAFT entries (1 oracle chunk + 2 FAISS
distractor chunks from a *different* contract, per the spec's "lowest cosine
similarity" rule), and checkpoints every 50 samples to
`cuad_raw_synthetic.parquet`. Also carves out a disjoint 50-chunk held-out pool
reused by Phase 5.


In [8]:
# Phase 2: Synthetic Data Generation (Teacher Model)
try:
    logger.info("=== PHASE 2: Synthetic Data Generation (Teacher Model) ===")

    # ---- Deterministic, disjoint train/eval chunk sampling (seeded -> stable across resumes) ----
    candidate_chunks = chunks_df[chunks_df["text"].str.len() > 200].reset_index(drop=True)
    total_needed = min(N_SYNTHETIC_SAMPLES + N_EVAL_SAMPLES, len(candidate_chunks))
    all_sampled = candidate_chunks.sample(n=total_needed, random_state=SEED).reset_index(drop=True)
    sample_order = all_sampled.iloc[: min(N_SYNTHETIC_SAMPLES, total_needed)].reset_index(drop=True)
    eval_sample_order = all_sampled.iloc[min(N_SYNTHETIC_SAMPLES, total_needed):].reset_index(drop=True)
    logger.info(f"Training sample pool: {len(sample_order)} chunks. Held-out eval pool: {len(eval_sample_order)} chunks.")

    TEACHER_PROMPT_TEMPLATE = (
        "You are an expert legal analyst. Read the following contract section and produce "
        "1 highly specific, professional legal question and its exact factual answer based "
        "strictly on the text.\n\nContract Excerpt:\n{oracle_chunk}\n\n"
        "Output Format:\nQUESTION: <question>\nANSWER: <exact answer>"
    )

    def generate_qa(oracle_text, model, tokenizer):
        prompt = TEACHER_PROMPT_TEMPLATE.format(oracle_chunk=oracle_text)
        messages = [{"role": "user", "content": prompt}]
        # input_ids = tokenizer.apply_chat_template(
        #     messages, add_generation_prompt=True, return_tensors="pt"
        # ).to(model.device)

        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
        input_ids = inputs["input_ids"].to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=300,
                do_sample=True,
                temperature=0.3,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
        return generated

    def parse_qa(generated_text):
        question, answer = None, None
        for line in generated_text.splitlines():
            line = line.strip()
            if line.upper().startswith("QUESTION:"):
                question = line.split(":", 1)[1].strip()
            elif line.upper().startswith("ANSWER:"):
                answer = line.split(":", 1)[1].strip()
        return question, answer

    def get_distractors(oracle_row, k=N_DISTRACTORS):
        '''Per spec: distractors are the LOWEST cosine-similarity chunks from a
        different contract than the oracle (i.e. random/easy negatives, matching
        the original RAFT paper's use of unrelated distractor documents).'''
        oracle_emb = np.asarray(oracle_row["embedding"], dtype=np.float32).reshape(1, -1)
        sims, idxs = faiss_index.search(oracle_emb, faiss_index.ntotal)
        candidates = []
        for sim, idx in zip(sims[0][::-1], idxs[0][::-1]):  # ascending similarity
            cand_row = chunks_df.iloc[idx]
            if cand_row["source_doc_id"] != oracle_row["source_doc_id"]:
                candidates.append(cand_row)
            if len(candidates) >= k:
                break
        return candidates

    # ---- Resume: pull existing progress, if any ----
    if is_phase_complete(RAW_SYNTHETIC_FILE):
        existing_synth_df = pull_parquet(RAW_SYNTHETIC_FILE)
    else:
        existing_synth_df = pd.DataFrame(
            columns=["sample_idx", "question", "answer", "oracle_chunk_id", "oracle_text",
                     "distractor_chunk_ids", "distractor_texts"]
        )

    start_idx = len(existing_synth_df)

    if start_idx >= len(sample_order):
        logger.info("Phase 2 synthetic generation already complete. Skipping teacher load.")
        raw_synthetic_df = existing_synth_df
    else:
        logger.info(f"Resuming generation from sample {start_idx}/{len(sample_order)}.")
        from transformers import AutoModelForCausalLM, AutoTokenizer

        teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
        teacher_model = AutoModelForCausalLM.from_pretrained(
            TEACHER_MODEL_NAME,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        teacher_model.eval()

        new_records = []
        for i in range(start_idx, len(sample_order)):
            oracle_row = sample_order.iloc[i]
            try:
                raw_output = generate_qa(oracle_row["text"], teacher_model, teacher_tokenizer)
                question, answer = parse_qa(raw_output)
                if not question or not answer:
                    logger.warning(f"Sample {i}: could not parse QUESTION/ANSWER, skipping.")
                    continue
                distractors = get_distractors(oracle_row)
                new_records.append({
                    "sample_idx": i,
                    "question": question,
                    "answer": answer,
                    "oracle_chunk_id": int(oracle_row["chunk_id"]),
                    "oracle_text": oracle_row["text"],
                    "distractor_chunk_ids": [int(d["chunk_id"]) for d in distractors],
                    "distractor_texts": [d["text"] for d in distractors],
                })
            except Exception as sample_err:
                logger.warning(f"Sample {i} generation failed: {sample_err}")
                continue

            is_last = (i == len(sample_order) - 1)
            if (i + 1) % BATCH_CHECKPOINT_EVERY == 0 or is_last:
                if new_records:
                    combined_df = pd.concat([existing_synth_df, pd.DataFrame(new_records)], ignore_index=True)
                else:
                    combined_df = existing_synth_df
                push_parquet(combined_df, RAW_SYNTHETIC_FILE)
                existing_synth_df = combined_df
                new_records = []
                logger.info(f"Checkpoint saved at sample {i + 1}/{len(sample_order)}.")

        raw_synthetic_df = existing_synth_df

        del teacher_model, teacher_tokenizer
        clear_cuda_cache_and_log()

    logger.info(f"Phase 2 complete: {len(raw_synthetic_df)} raw synthetic RAFT samples available.")

except Exception:
    logger.error("Phase 2 failed.")
    logger.error(traceback.format_exc())
    raise

2026-07-25 19:44:42,602 | INFO | === PHASE 2: Synthetic Data Generation (Teacher Model) ===
2026-07-25 19:44:42,619 | INFO | Training sample pool: 500 chunks. Held-out eval pool: 50 chunks.
2026-07-25 19:44:42,794 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_raw_synthetic.parquet "HTTP/1.1 404 Not Found"
2026-07-25 19:44:42,796 | INFO | Resuming generation from sample 0/500.
2026-07-25 19:44:53,356 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/gemma-2-9b-it-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-25 19:44:53,460 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/gemma-2-9b-it-bnb-4bit/27b027bcbb6b1861b02551d5c699d5d07f29610a/config.json "HTTP/1.1 200 OK"
2026-07-25 19:44:53,566 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/gemma-2-9b-it-bnb-4bit/27b027bcbb6b1861b02551d5c699d5d07f29610a/config.json "HTTP/1.1 

config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

2026-07-25 19:44:53,654 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/gemma-2-9b-it-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-25 19:44:53,757 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/gemma-2-9b-it-bnb-4bit/27b027bcbb6b1861b02551d5c699d5d07f29610a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-25 19:44:53,861 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/gemma-2-9b-it-bnb-4bit/27b027bcbb6b1861b02551d5c699d5d07f29610a/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

2026-07-25 19:44:53,952 | INFO | HTTP Request: GET https://huggingface.co/api/models/unsloth/gemma-2-9b-it-bnb-4bit/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-25 19:44:54,028 | INFO | HTTP Request: GET https://huggingface.co/api/models/unsloth/gemma-2-9b-it-bnb-4bit/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-07-25 19:44:54,176 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/gemma-2-9b-it-bnb-4bit/resolve/main/tokenizer.json "HTTP/1.1 302 Found"


...

Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

2026-07-25 21:38:42,064 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-pipeline/preupload/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

2026-07-25 21:38:43,468 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-pipeline/commit/main "HTTP/1.1 200 OK"
2026-07-25 21:38:43,469 | INFO | Pushed cuad_raw_synthetic.parquet (500 rows) to abhifdsdf/cuad-raft-pipeline
2026-07-25 21:38:43,470 | INFO | Checkpoint saved at sample 500/500.
2026-07-25 21:38:44,038 | INFO |   GPU 0: 15.47 GB free / 15.64 GB total
2026-07-25 21:38:44,040 | INFO |   GPU 1: 15.48 GB free / 15.64 GB total
2026-07-25 21:38:44,041 | INFO | Memory cleanup complete.
2026-07-25 21:38:44,041 | INFO | Phase 2 complete: 500 raw synthetic RAFT samples available.


## PHASE 3: Automated Grounding Filter (Cross-Encoder)

Scores each `(oracle_chunk, synthetic_answer)` pair with `BAAI/bge-reranker-base`
and drops anything below the 0.5 grounding threshold, to keep hallucinated
answers out of the fine-tuning set.


In [9]:
# Phase 3: Automated Grounding Filter (Cross-Encoder)
try:
    logger.info("=== PHASE 3: Automated Grounding Filter ===")

    if is_phase_complete(FILTERED_RAFT_FILE):
        logger.info(f"{FILTERED_RAFT_FILE} found on {HF_REPO_ID}. Skipping grounding filter.")
        filtered_df = pull_parquet(FILTERED_RAFT_FILE)
    else:
        raw_df = pull_parquet(RAW_SYNTHETIC_FILE) if is_phase_complete(RAW_SYNTHETIC_FILE) else raw_synthetic_df.copy()

        from sentence_transformers import CrossEncoder

        reranker = CrossEncoder(RERANKER_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
        pairs = list(zip(raw_df["oracle_text"].tolist(), raw_df["answer"].tolist()))
        raw_scores = reranker.predict(pairs, show_progress_bar=True)

        # bge-reranker-base outputs an unbounded relevance logit; squash to [0,1]
        # with a sigmoid so the 0.5 threshold from the spec is meaningful.
        grounding_scores = 1.0 / (1.0 + np.exp(-np.asarray(raw_scores, dtype=np.float64)))
        raw_df = raw_df.copy()
        raw_df["grounding_score"] = grounding_scores

        filtered_df = raw_df[raw_df["grounding_score"] >= GROUNDING_THRESHOLD].reset_index(drop=True)
        logger.info(
            f"Grounding filter kept {len(filtered_df)}/{len(raw_df)} samples "
            f"(threshold={GROUNDING_THRESHOLD})."
        )

        push_parquet(filtered_df, FILTERED_RAFT_FILE)

        del reranker
        clear_cuda_cache_and_log()

    logger.info(f"Phase 3 complete: {len(filtered_df)} grounded RAFT samples ready for fine-tuning.")

except Exception:
    logger.error("Phase 3 failed.")
    logger.error(traceback.format_exc())
    raise


2026-07-25 21:38:44,050 | INFO | === PHASE 3: Automated Grounding Filter ===
2026-07-25 21:38:44,267 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_filtered_raft.parquet "HTTP/1.1 404 Not Found"
2026-07-25 21:38:44,342 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_raw_synthetic.parquet "HTTP/1.1 302 Found"
2026-07-25 21:38:44,418 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_raw_synthetic.parquet "HTTP/1.1 302 Found"


cuad_raw_synthetic.parquet: reconstructing file:   0%|          |  0.00B /  756kB            

cuad_raw_synthetic.parquet: downloading bytes:           |  0.00B            

2026-07-25 21:38:45,413 | INFO | Pulled cuad_raw_synthetic.parquet (500 rows) from abhifdsdf/cuad-raft-pipeline
2026-07-25 21:38:52,480 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-25 21:38:52,482 | INFO | No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-07-25 21:38:52,555 | INFO | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-07-25 21:38:52,628 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-25 21:38:52,741 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-07-25 21:38:52,858 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

2026-07-25 21:39:17,294 | INFO | Grounding filter kept 500/500 samples (threshold=0.5).
2026-07-25 21:39:17,419 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-pipeline/preupload/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

2026-07-25 21:39:18,464 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-pipeline/commit/main "HTTP/1.1 200 OK"
2026-07-25 21:39:18,465 | INFO | Pushed cuad_filtered_raft.parquet (500 rows) to abhifdsdf/cuad-raft-pipeline
2026-07-25 21:39:19,015 | INFO |   GPU 0: 15.47 GB free / 15.64 GB total
2026-07-25 21:39:19,016 | INFO |   GPU 1: 15.48 GB free / 15.64 GB total
2026-07-25 21:39:19,017 | INFO | Memory cleanup complete.
2026-07-25 21:39:19,017 | INFO | Phase 3 complete: 500 grounded RAFT samples ready for fine-tuning.


## PHASE 4: Student Model Fine-Tuning (Unsloth QLoRA)

Fine-tunes `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` on the grounded RAFT dataset
with Unsloth (single-GPU, free-tier compatible) and pushes the LoRA adapter to
the Hub.


In [10]:
# Phase 4: Student Model Fine-Tuning (Unsloth QLoRA)
try:
    logger.info("=== PHASE 4: Student Model Fine-Tuning (Unsloth QLoRA) ===")

    adapter_already_pushed = is_phase_complete("adapter_config.json", repo_id=MODEL_REPO_ID, repo_type="model")

    if adapter_already_pushed:
        logger.info(f"adapter_config.json already present at {MODEL_REPO_ID}. Skipping fine-tuning.")
    else:
        filtered_df = pull_parquet(FILTERED_RAFT_FILE) if is_phase_complete(FILTERED_RAFT_FILE) else filtered_df.copy()
        filtered_df = filtered_df.reset_index(drop=True)

        import unsloth  # must be imported before transformers/peft/trl for Unsloth's patches to apply
        from unsloth import FastLanguageModel
        from trl import SFTTrainer, SFTConfig
        from datasets import Dataset

        student_model, student_tokenizer = FastLanguageModel.from_pretrained(
            model_name=STUDENT_MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
        )
        student_model = FastLanguageModel.get_peft_model(
            student_model,
            r=16,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=32,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=SEED,
        )

        def build_chatml_example(row):
            context_chunks = [row["oracle_text"]] + list(row["distractor_texts"])
            rng = random.Random(int(row["sample_idx"]))
            rng.shuffle(context_chunks)
            context_block = "\n\n---\n\n".join(context_chunks)
            user_content = f"Context:\n{context_block}\n\nQuestion: {row['question']}"
            messages = [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": row["answer"]},
            ]
            return student_tokenizer.apply_chat_template(messages, tokenize=False)

        filtered_df["text"] = filtered_df.apply(build_chatml_example, axis=1)
        train_dataset = Dataset.from_pandas(filtered_df[["text"]])

        sft_config = SFTConfig(
            output_dir=os.path.join(WORKDIR, "qlora_outputs"),
            dataset_text_field="text",
            max_length=MAX_SEQ_LENGTH,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,   # effective batch size = 16
            num_train_epochs=2,
            learning_rate=2e-4,
            warmup_ratio=0.05,
            fp16=True,
            bf16=False,
            optim="adamw_8bit",
            logging_steps=10,
            save_strategy="no",
            seed=SEED,
            report_to=[],
        )

        trainer = SFTTrainer(
            model=student_model,
            train_dataset=train_dataset,
            processing_class=student_tokenizer,
            args=sft_config,
        )
        train_result = trainer.train()
        logger.info(f"Training finished. Final loss: {train_result.training_loss:.4f}")

        student_model.push_to_hub(MODEL_REPO_ID, token=hf_token)
        student_tokenizer.push_to_hub(MODEL_REPO_ID, token=hf_token)
        logger.info(f"Pushed fine-tuned LoRA adapter to {MODEL_REPO_ID}")

        del trainer, student_model, student_tokenizer
        clear_cuda_cache_and_log()

    logger.info("Phase 4 complete.")

except Exception:
    logger.error("Phase 4 failed.")
    logger.error(traceback.format_exc())
    raise


2026-07-25 21:39:19,030 | INFO | === PHASE 4: Student Model Fine-Tuning (Unsloth QLoRA) ===
2026-07-25 21:39:19,116 | INFO | HTTP Request: HEAD https://huggingface.co/abhifdsdf/qwen2.5-3b-cuad-raft/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-25 21:39:19,196 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_filtered_raft.parquet "HTTP/1.1 302 Found"
2026-07-25 21:39:19,272 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_filtered_raft.parquet "HTTP/1.1 302 Found"


cuad_filtered_raft.parquet: reconstructing file:   0%|          |  0.00B /  761kB            

cuad_filtered_raft.parquet: downloading bytes:           |  0.00B            

2026-07-25 21:39:19,992 | INFO | Pulled cuad_filtered_raft.parquet (500 rows) from abhifdsdf/cuad-raft-pipeline
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpu82pagkh/tokenizer_config.json.


2026-07-25 22:01:06,009 | INFO | HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-3B-Instruct-bnb-4bit "HTTP/1.1 200 OK"
2026-07-25 22:01:06,074 | INFO | HTTP Request: POST https://huggingface.co/api/validate-yaml "HTTP/1.1 200 OK"
2026-07-25 22:01:06,162 | INFO | HTTP Request: POST https://huggingface.co/api/models/abhifdsdf/qwen2.5-3b-cuad-raft/preupload/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

2026-07-25 22:01:07,411 | INFO | HTTP Request: POST https://huggingface.co/api/models/abhifdsdf/qwen2.5-3b-cuad-raft/commit/main "HTTP/1.1 200 OK"
2026-07-25 22:01:07,415 | INFO | Pushed fine-tuned LoRA adapter to abhifdsdf/qwen2.5-3b-cuad-raft
2026-07-25 22:01:07,926 | INFO |   GPU 0: 13.33 GB free / 15.64 GB total
2026-07-25 22:01:07,927 | INFO |   GPU 1: 15.48 GB free / 15.64 GB total
2026-07-25 22:01:07,928 | INFO | Memory cleanup complete.
2026-07-25 22:01:07,929 | INFO | Phase 4 complete.


## PHASE 5: Evaluation & Comparison Table

Builds a 50-question held-out set (disjoint chunk pool from Phase 2), evaluates
Base Zero-Shot, Base+RAG, and Fine-Tuned RAFT at retrieval depths k=1/3/5, and
scores Ragas Faithfulness/Answer Relevancy using the **teacher model reloaded as
an independent judge** (not the fine-tuned student itself).


In [11]:
# Phase 5: Evaluation & Comparison Table
try:
    logger.info("=== PHASE 5: Evaluation & Comparison Table ===")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    from sentence_transformers import SentenceTransformer
    from peft import PeftModel

    def retrieve_topk(question, k, embed_model):
        q_emb = embed_model.encode([question], normalize_embeddings=True).astype(np.float32)
        _, idxs = faiss_index.search(q_emb, k)
        return [chunks_df.iloc[idx]["text"] for idx in idxs[0]]

    def generate_answer(model, tokenizer, question, context_chunks=None, max_new_tokens=150):
        if context_chunks:
            context_block = "\n\n---\n\n".join(context_chunks)
            user_content = (
                f"Context:\n{context_block}\n\nQuestion: {question}\n"
                "Answer concisely and factually based only on the context."
            )
        else:
            user_content = f"Question: {question}\nAnswer concisely and factually."
        messages = [{"role": "user", "content": user_content}]
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            out_ids = model.generate(
                input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()

    # ---- 5a. Teacher reloaded for (a) eval-set generation if needed, (b) independent Ragas judge ----
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
    teacher_model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL_NAME, device_map="auto", torch_dtype=torch.float16,
    )
    teacher_model.eval()
    logger.info("Teacher model reloaded (held-out set generation + Ragas judge duty).")

    if is_phase_complete(EVAL_HOLDOUT_FILE):
        eval_df = pull_parquet(EVAL_HOLDOUT_FILE)
    else:
        eval_records = []
        for i, row in eval_sample_order.iterrows():
            try:
                raw_output = generate_qa(row["text"], teacher_model, teacher_tokenizer)
                question, answer = parse_qa(raw_output)
                if not question or not answer:
                    logger.warning(f"Eval sample {i}: could not parse QUESTION/ANSWER, skipping.")
                    continue
                eval_records.append({
                    "eval_idx": int(i),
                    "question": question,
                    "gold_answer": answer,
                    "oracle_chunk_id": int(row["chunk_id"]),
                })
            except Exception as eval_err:
                logger.warning(f"Eval sample {i} generation failed: {eval_err}")
        eval_df = pd.DataFrame(eval_records)
        push_parquet(eval_df, EVAL_HOLDOUT_FILE)

    logger.info(f"Held-out evaluation set ready: {len(eval_df)} questions.")

    # ---- 5b. Shared retrieval (model-independent) at k = 1, 3, 5 ----
    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
    for k in (1, 3, 5):
        eval_df[f"ctx_k{k}"] = eval_df["question"].apply(lambda q, kk=k: retrieve_topk(q, kk, embed_model))
    del embed_model
    clear_cuda_cache_and_log()

    # ---- 5c. Config A: Base Student, Zero-Shot ----
    base_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)
    base_model = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL_NAME, device_map="auto", torch_dtype=torch.float16,
    )
    base_model.eval()

    eval_df["pred_zero_shot"] = [
        generate_answer(base_model, base_tokenizer, q, context_chunks=None) for q in eval_df["question"]
    ]
    logger.info("Base Zero-Shot predictions complete.")

    # ---- 5d. Config B: Base Student + RAG (k = 1, 3, 5) ----
    for k in (1, 3, 5):
        eval_df[f"pred_rag_k{k}"] = [
            generate_answer(base_model, base_tokenizer, q, context_chunks=ctx)
            for q, ctx in zip(eval_df["question"], eval_df[f"ctx_k{k}"])
        ]
    logger.info("Base + RAG predictions complete.")

    del base_model, base_tokenizer
    clear_cuda_cache_and_log()

    # ---- 5e. Config C: Fine-Tuned RAFT Student (k = 1, 3, 5) ----
    raft_tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO_ID)
    raft_base_model = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL_NAME, device_map="auto", torch_dtype=torch.float16,
    )
    raft_model = PeftModel.from_pretrained(raft_base_model, MODEL_REPO_ID)
    raft_model.eval()

    for k in (1, 3, 5):
        eval_df[f"pred_raft_k{k}"] = [
            generate_answer(raft_model, raft_tokenizer, q, context_chunks=ctx)
            for q, ctx in zip(eval_df["question"], eval_df[f"ctx_k{k}"])
        ]
    logger.info("Fine-Tuned RAFT predictions complete.")

    del raft_model, raft_base_model, raft_tokenizer
    clear_cuda_cache_and_log()

    # ---- 5f. Exact Match / Span F1 ----
    metrics = {
        "Base Model (Zero-Shot)": {},
        "Base Model + RAG": {},
        "Fine-Tuned RAFT Model": {},
    }

    def mean_em_f1(pred_col, gold_col="gold_answer"):
        ems = [exact_match(p, g) for p, g in zip(eval_df[pred_col], eval_df[gold_col])]
        f1s = [f1_score(p, g) for p, g in zip(eval_df[pred_col], eval_df[gold_col])]
        return float(np.mean(ems)), float(np.mean(f1s))

    rank_rows = ["Exact Match @ Rank-1", "Exact Match @ Rank-3", "Exact Match @ Rank-5"]

    # Zero-shot never sees retrieved context, so its "rank" scores are identical
    # across k -- reported once and repeated, as noted in the intro markdown cell.
    em_zs, f1_zs = mean_em_f1("pred_zero_shot")
    for row in rank_rows:
        metrics["Base Model (Zero-Shot)"][row] = em_zs
    metrics["Base Model (Zero-Shot)"]["Span F1 Score"] = f1_zs

    for label, prefix in [("Base Model + RAG", "pred_rag"), ("Fine-Tuned RAFT Model", "pred_raft")]:
        for k, row in zip((1, 3, 5), rank_rows):
            em, _ = mean_em_f1(f"{prefix}_k{k}")
            metrics[label][row] = em
        _, f1_at_3 = mean_em_f1(f"{prefix}_k3")
        metrics[label]["Span F1 Score"] = f1_at_3

    # ---- 5g. Ragas Faithfulness / Answer Relevancy (teacher as independent judge) ----
    try:
        from datasets import Dataset as HFDataset
        from transformers import pipeline as hf_pipeline
        from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
        from ragas import evaluate as ragas_evaluate
        from ragas import EvaluationDataset
        from ragas.metrics import Faithfulness, AnswerRelevancy
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper

        judge_text_gen_pipe = hf_pipeline(
            "text-generation",
            model=teacher_model,
            tokenizer=teacher_tokenizer,
            max_new_tokens=300,
            do_sample=False,
        )
        judge_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=judge_text_gen_pipe))
        judge_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME))

        def compute_ragas_scores(answer_col):
            ragas_hf_ds = HFDataset.from_dict({
                "user_input": eval_df["question"].tolist(),
                "response": eval_df[answer_col].tolist(),
                "retrieved_contexts": eval_df["ctx_k3"].tolist(),
            })
            ragas_eval_ds = EvaluationDataset.from_hf_dataset(ragas_hf_ds)
            result = ragas_evaluate(
                dataset=ragas_eval_ds,
                metrics=[Faithfulness(), AnswerRelevancy()],
                llm=judge_llm,
                embeddings=judge_embeddings,
            )
            result_df = result.to_pandas()
            return float(result_df["faithfulness"].mean()), float(result_df["answer_relevancy"].mean())

        for label, answer_col in [
            ("Base Model (Zero-Shot)", "pred_zero_shot"),
            ("Base Model + RAG", "pred_rag_k3"),
            ("Fine-Tuned RAFT Model", "pred_raft_k3"),
        ]:
            try:
                faith, rel = compute_ragas_scores(answer_col)
            except Exception as ragas_err:
                logger.warning(f"Ragas scoring failed for '{label}': {ragas_err}")
                faith, rel = float("nan"), float("nan")
            metrics[label]["Ragas Faithfulness"] = faith
            metrics[label]["Ragas Answer Relevancy"] = rel

    except Exception:
        logger.warning("Ragas evaluation block failed; filling NaN so the notebook still completes.")
        logger.warning(traceback.format_exc())
        for label in metrics:
            metrics[label].setdefault("Ragas Faithfulness", float("nan"))
            metrics[label].setdefault("Ragas Answer Relevancy", float("nan"))

    del teacher_model, teacher_tokenizer
    clear_cuda_cache_and_log()

    # ---- 5h. Assemble & print comparison table ----
    row_order = rank_rows + ["Span F1 Score", "Ragas Faithfulness", "Ragas Answer Relevancy"]
    col_order = ["Base Model (Zero-Shot)", "Base Model + RAG", "Fine-Tuned RAFT Model"]

    comparison_df = pd.DataFrame(index=row_order, columns=col_order, dtype=float)
    for col in col_order:
        for row in row_order:
            comparison_df.loc[row, col] = metrics[col][row]
    comparison_df.index.name = "Evaluation Metric"

    print(comparison_df.reset_index().to_markdown(index=False, floatfmt=".4f"))

    local_csv_path = os.path.join(WORKDIR, EVAL_RESULTS_FILE)
    comparison_df.reset_index().to_csv(local_csv_path, index=False)
    api.upload_file(
        path_or_fileobj=local_csv_path,
        path_in_repo=EVAL_RESULTS_FILE,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        commit_message="Add evaluation_results.csv",
    )
    logger.info(f"Pushed {EVAL_RESULTS_FILE} to {HF_REPO_ID}")
    logger.info("Phase 5 complete. RAFT pipeline finished end-to-end.")

except Exception:
    logger.error("Phase 5 failed.")
    logger.error(traceback.format_exc())
    raise


2026-07-25 22:01:07,956 | INFO | === PHASE 5: Evaluation & Comparison Table ===
2026-07-25 22:01:08,025 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/gemma-2-9b-it-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-25 22:01:08,041 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/gemma-2-9b-it-bnb-4bit/27b027bcbb6b1861b02551d5c699d5d07f29610a/config.json "HTTP/1.1 200 OK"
2026-07-25 22:01:08,114 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/gemma-2-9b-it-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-25 22:01:08,132 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/gemma-2-9b-it-bnb-4bit/27b027bcbb6b1861b02551d5c699d5d07f29610a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-25 22:01:08,207 | INFO | HTTP Request: GET https://huggingface.co/api/models/unsloth/gemma-2-9b-it-bnb-4bit/tree/main/additional_chat_templates?recursive=fals

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

2026-07-25 22:01:13,058 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/gemma-2-9b-it-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-25 22:01:13,076 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/gemma-2-9b-it-bnb-4bit/27b027bcbb6b1861b02551d5c699d5d07f29610a/generation_config.json "HTTP/1.1 200 OK"
2026-07-25 22:01:13,316 | INFO | Teacher model reloaded (held-out set generation + Ragas judge duty).
2026-07-25 22:01:13,393 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_eval_holdout.parquet "HTTP/1.1 404 Not Found"
2026-07-25 22:01:13,405 | WARNING | Eval sample 0 generation failed: too many indices for tensor of dimension 2
2026-07-25 22:01:13,408 | WARNING | Eval sample 1 generation failed: too many indices for tensor of dimension 2
2026-07-25 22:01:13,410 | WARNING | Eval sample 2 generation failed: too many indices for tensor of dimensio

/tmp/ipykernel_185/2021363591.py:30: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  input_ids = inputs["input_ids"].to(model.device)


2026-07-25 22:01:13,656 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-pipeline/preupload/main "HTTP/1.1 200 OK"


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2026-07-25 22:01:19,720 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-25 22:01:19,832 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-25 22:01:19,935 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-07-25 22:01:20,039 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-07-25 22:01:20,124 | INFO | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-07-25 22:01:20,163 | ERROR | Phase 5 failed.
2026-07-25 22:01:20,166 | ERROR | Traceback (most recent call last):
  File "/tmp/ipykernel_185/3257619651.py", line 69, in <cell line: 0>
    eval_df[f"ctx_k{k}"] = eval_df["question"].apply(lambda q, kk=k: retrieve_topk(q, kk, embed_model))
                           ~~~~~~~^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 4378, in __getitem__
    indexer = self.columns.get_loc(key)
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/range.py", line 525, in get_loc
    raise KeyError(key)
KeyError: 'question'



KeyError: 'question'